# Библиотеки

In [7]:
from pprint import pprint

import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import seaborn as sns

## 11.16 Многослойный перцептрон MLP: с учителем

- Age (Возраст в годах)
- Ed (Образование в уровнях: 1 – неоконченное среднее, 2 – среднее, 3- среднее-специальное, 4- неоконченное высшее, 5 – высшее) - это порядковая переменная!
- Empl (Сколько лет работает на текущем месте работы)
- Adress (Сколько лет проживает по текущему адресу)
- Inc (Сколько лет проживает по текущему адресу)
- DebtInc (Соотношение имеющейся задолженности к доходам (x100))
- C_debt (Долги по кредиткам в тыс у.е.)
- O_debt (Другие долги)
- F_debt (Не возвращал кредит (были факты невозврата) =1 или Нет фактор невозврата = 0) - эту переменную и будем предсказывать (вернет или нет заемщик кредит).

In [3]:
d = pd.read_excel('./data/MLP.xlsx')

In [4]:
d

,Age,Ed,Empl,Adress,Inc,DebtInc,C_debt,O_debt,F_debt
0,41,3,17,12,176,9.3,11.36,5.01,1
1,27,1,10,6,31,17.3,1.36,4.00,0
2,40,1,15,14,55,5.5,0.86,2.17,0
3,41,1,15,14,120,2.9,2.66,0.82,0
4,24,2,2,0,28,17.3,1.79,3.06,1
...,...,...,...,...,...,...,...,...,...
695,36,2,6,15,27,4.6,0.26,0.98,1
696,29,2,6,4,21,11.5,0.37,2.05,0
697,33,1,15,3,32,7.6,0.49,1.94,0
698,45,1,19,22,77,8.4,2.30,4.17,0


In [5]:
X = d.drop('F_debt', axis=1)

y = d['F_debt']

In [8]:
X_s = StandardScaler().fit_transform(X)

In [9]:
X_o, X_t, y_o, y_t = train_test_split(X_s, y, test_size=0.3, stratify=y)

Настройте спецификацию модели многослойного перцептрона в переменной mlp с таким параметрами: 

- hidden_layer_sizes так чтобы было 2 слоя нейронов (потому что по умолчанию 100 нейронов в одном слое используется). Используйте степени двойки (32, 64, 128): первый слой должен быть минимум раза в три больше чем количество переменных – 64 Вам подойдет.

- max_iter=1000 увеличение кол-ва итераций до 1000, потому что может не найти наилучшее решение за 200 по умолчанию

- early_stopping=True чтобы оно прекратило "мучиться", если найдет раньше

In [10]:
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, early_stopping=True)

In [11]:
mlp.fit(X_o, y_o)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",1000
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",None


In [12]:
ypred = mlp.predict(X_t)

In [13]:
print(classification_report(y_t, ypred))

              precision    recall  f1-score   support

           0       0.80      0.93      0.86       155
           1       0.63      0.35      0.45        55

    accuracy                           0.78       210
   macro avg       0.72      0.64      0.65       210
weighted avg       0.76      0.78      0.75       210



In [14]:
confusion_matrix(y_t, ypred)

array([[144,  11],
       [ 36,  19]])

In [21]:
print(
    roc_auc_score(y_t, mlp.predict_proba(X_t)[:,1])
)

0.7797067448680352
